# Paper Stats

In [77]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import xarray as xr
import pandas as pd
import geopandas as gpd
from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils, isku_utils, plotting_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)
importlib.reload(plotting_utils)

<module 'utils.plotting_utils' from '/home/emily_zuetell/projects/poreallas/utils/plotting_utils.py'>

In [ ]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]


In [ ]:
# Set Consistent Quantiles Here
QUANTILES = [0.17, 0.83]

# Baseline period for Impact
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)
BASELINE_PERIOD = analysis_utils.get_baseline_period(effect, years=30)
# Define Forecast Months
FC_MONTHS = [10, 11, 12, 1, 2, 3]

config_total = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=False,
    age_weight=True,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

config_rate = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=True,
    age_weight=True,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

In [11]:
impact_total = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config_total.dims}),
    config_total,
    ensemble=True,
).sel(month=config_total.months)

impact_rate = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config_rate.dims}),
    config_rate,
    ensemble=True,
).sel(month=config_rate.months)

## Global Total

In [ ]:
# pop_weighted global impact (Sum of regions)
global_impact = analysis_utils.compute_global_impact(
    impact_total, config_total.socioeconomics, config_total.rate, group_dim="region"
)
# Mean and quantiels across ensemble members and samples
mean = global_impact.sum(dim="month").mean(dim=config_total.dims).values
q = global_impact.sum(dim="month").quantile(QUANTILES, dim=config_total.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


372932.70 (146931.55, 625836.05)


## Global Rate

In [ ]:
# pop_weighted global impact (weight rate by region pop)
global_impact = analysis_utils.compute_global_impact(
    impact_rate, config_rate.socioeconomics, config_rate.rate, group_dim="region"
)
# Mean and quantiels across ensemble members and samples
mean = global_impact.sum(dim="month").mean(dim=config_rate.dims).values
q = global_impact.sum(dim="month").quantile(QUANTILES, dim=config_rate.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


4.54 (1.79, 7.61)


## Global Rate Change

In [ ]:
# Compute RAW pop-weighted EFFECT as a RATE
# hotonly forecast
rate_fc = analysis_utils.pop_weight_sum(effect['/forecast_hotonly'], config_rate, impact=False)
# hotonly baseline
rate_baseline = analysis_utils.pop_weight_sum(effect['/baseline_hotonly'], config_rate, impact=False)

In [ ]:
# Compare forecast to monthly baseline mean
fc = rate_fc['effect'].groupby('time.month').mean() # Shouldn't do anything because forecast is already monthly
baseline = rate_baseline['effect'].groupby('time.month').mean() # Averages across years in baseline period
# Compute global rate (pop-weigted region avg)
fc_global = analysis_utils.compute_global_impact(fc, config_rate.socioeconomics, config_rate.rate, group_dim="region")
baseline_global = analysis_utils.compute_global_impact(baseline, config_rate.socioeconomics, config_rate.rate, group_dim="region")
# Compute percent change
pct_diff = ((fc_global-baseline_global)/baseline_global)*100

In [46]:
mean = pct_diff.sel(month = config_rate.months).sum(dim = 'month').mean(dim = config_rate.dims).values
q = pct_diff.sel(month = config_rate.months).sum(dim = 'month').quantile(QUANTILES, dim=config_rate.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

365.97 (135.76, 576.75)


## Rate by Income Group

In [ ]:
# Aggregate rate to ISO level
impact_rate_ISO, _, _ = analysis_utils.aggregate_impact(impact_rate, config_rate, "ISO")

In [ ]:
# Fetch country boundaries to align ISO with World Bank income group data (same as used in bar chart)
import cartopy.io.shapereader as shpreader

def fetch_country_boundaries():
    path = shpreader.natural_earth(resolution="110m", category="cultural", name="admin_0_countries")
    return gpd.read_file(path)[["NAME", "ADM0_A3", "geometry"]]


def join_to_countries(gdf, countries):
    return gpd.sjoin(
        gdf.to_crs(countries.crs), countries, how="left", predicate="intersects"
    )


countries = fetch_country_boundaries()

In [ ]:
# 6-month total
total = impact_rate_ISO.sum(dim="month")
# Mean, Quantiles across ensemble member, sample
df = (
    xr.Dataset(
        {
            "impact": total.mean(dim=config_total.dims),
            "p17": total.quantile(0.17, dim=config_total.dims).drop_vars("quantile"),
            "p83": total.quantile(0.83, dim=config_total.dims).drop_vars("quantile"),
        }
    )
    .to_dataframe()
    .reset_index()
)
# Clean up df for merge with income data
df = df.drop(columns=["year"])
iso_to_name = countries.set_index("ADM0_A3")["NAME"].to_dict()
df["country"] = df["ISO"].map(iso_to_name)
# S. Sudan
df.loc[df["ISO"] == "SSD", "country"] = "South Sudan"

# Load World Bank 2025 Income Classes
income = pd.read_csv("../data/base/income_class_2025.csv")
# Merge with Impacts
merged = df.merge(income, left_on="country", right_on="Economy", how="inner")

In [ ]:
df = merged
cols = ["impact", "p17", "p83"]

# Country avg (Is pop weighted needed?)
low_mid = df.loc[df["Income group"].isin(["Low income", "Lower middle income"]), cols].mean()
high = df.loc[df["Income group"] == "High income", cols].mean()

ratio = low_mid / high
# ratio_low = low_mid["p17"] / high["p83"] These ratios aren't very representative... leaving out for now
# ratio_high = low_mid["p83"] / high["p17"]

## Total Deaths by Country

In [ ]:
# Total deaths aggregated to ISO
impact_total_ISO, _, _ = analysis_utils.aggregate_impact(impact_total, config_total, "ISO")

In [ ]:
# India
iso = 'IND'
mean = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').mean(dim = config_rate.dims).values
q = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').quantile(QUANTILES, dim=config_rate.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

55213.18 (12732.14, 100433.74)


In [ ]:
# Indonesia
iso = 'IDN'
mean = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').mean(dim = config_rate.dims).values
q = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').quantile(QUANTILES, dim=config_rate.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

29278.99 (8685.27, 55098.20)


In [ ]:
# Brazil
iso = 'BRA'
mean = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').mean(dim = config_rate.dims).values
q = impact_total_ISO.sel(ISO = iso).sum(dim = 'month').quantile(QUANTILES, dim=config_rate.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

19478.49 (6949.58, 34655.38)


## Running Total

In [ ]:
# Load November running total data (Note: This includes everything we need AND the proper ERA5 method)
PROJ_MONTH = 11
MONTHS_RT = [
    (5 + i) % 12 + 1 for i in range((PROJ_MONTH - 6) % 12)]
RATE = False # Compute total deaths
config_RT = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}_RT",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=RATE,
    months=MONTHS_RT,  # Running Total Months (June-)
    hotonly="hotonly",
    dims=["sample"],  # No ensemble members in Running Total Data
)

In [109]:
## Running Total ##
effect_rt = xr.open_datatree("../data/v2611/v20261110_effects_rt_1hr.zarr")

# Insert baseline from 'effect' since baseline does not need to be computed in running total workflow
effect_rt["/baseline"] = effect["/baseline"]
effect_rt["/baseline_hotonly"] = effect["/baseline_hotonly"]
# Rename forecast for analysis functions
effect_rt["/forecast"] = effect_rt["/forecast_runningtotal"]
effect_rt["/forecast_hotonly"] = effect_rt["/forecast_hotonly_runningtotal"]

In [65]:
impact_rt = analysis_utils.compute_impact(
    effect_rt.chunk({dim: -1 for dim in ['sample']}),
    config_RT,
    ensemble=True,
).sel(month=config_RT.months)

In [68]:
mean = impact_rt.sel(month = [6, 7, 8, 9]).sum(dim = 'month').sum(dim = 'region').mean(dim = config_RT.dims).values
q = impact_rt.sel(month = [6, 7, 8, 9]).sum(dim = 'month').sum(dim = 'region').quantile(QUANTILES, dim=config_RT.dims).values
print(f"{mean:.2f} ({q[0]:.2f}, {q[1]:.2f})")

289931.06 (153244.53, 444485.67)
